# RadGraph-XL Data Visualization

This notebook creates aggregate figures for the dataset section. It reads the credentialed ZIP but does not display report text.

In [ ]:
from __future__ import annotations

import json
import os
import zipfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / ".env").exists() and (PROJECT_ROOT.parent / ".env").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
load_dotenv(PROJECT_ROOT / ".env")

zip_path = Path(os.environ["RADGRAPH_XL_ZIP"])
figure_dir = PROJECT_ROOT / "outputs" / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
print(zip_path)

In [ ]:
with zipfile.ZipFile(zip_path) as archive:
    jsonl_members = [name for name in archive.namelist() if name.lower().endswith(".jsonl")]
    assert len(jsonl_members) == 1, jsonl_members
    with archive.open(jsonl_members[0]) as handle:
        records = [json.loads(line) for line in handle if line.strip()]

print({"records": len(records), "jsonl_member": jsonl_members[0]})

In [ ]:
def flatten_entities(record):
    return [entity for sentence_entities in record["ner"] for entity in sentence_entities]


def flatten_relations(record):
    return [relation for sentence_relations in record["relations"] for relation in sentence_relations]


def token_count(record):
    return sum(len(sentence) for sentence in record["sentences"])


def relation_gap(relation):
    source_start, source_end, target_start, target_end, _label = relation
    return max(0, max(target_start - source_end, source_start - target_end))


report_df = pd.DataFrame([
    {
        "doc_key": record["doc_key"],
        "dataset": record["dataset"],
        "token_count": token_count(record),
        "entity_count": len(flatten_entities(record)),
        "relation_count": len(flatten_relations(record)),
    }
    for record in records
])

entity_df = pd.DataFrame([
    {
        "dataset": record["dataset"],
        "label": entity[2],
        "category": entity[2].split("::", maxsplit=1)[0],
        "assertion": entity[2].split("::", maxsplit=1)[1],
        "span_length": entity[1] - entity[0] + 1,
    }
    for record in records
    for entity in flatten_entities(record)
])

relation_df = pd.DataFrame([
    {
        "dataset": record["dataset"],
        "label": relation[4],
        "token_gap": relation_gap(relation),
    }
    for record in records
    for relation in flatten_relations(record)
])

report_df.head()

In [ ]:
saved_figures = {}


def save_current_figure(name):
    path = figure_dir / name
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.show()
    plt.close()
    saved_figures[name] = str(path)
    return path

In [ ]:
plt.figure(figsize=(7, 4))
dataset_counts = report_df["dataset"].value_counts().sort_index()
sns.barplot(x=dataset_counts.index, y=dataset_counts.values, color="#4c78a8")
plt.title("Reports by RadGraph-XL MIMIC Subset")
plt.xlabel("Modality subset")
plt.ylabel("Report count")
plt.xticks(rotation=20, ha="right")
save_current_figure("modality_distribution.png")

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(report_df["token_count"], bins=18, color="#59a14f")
plt.axvline(384, color="#f28e2b", linestyle="--", label="384 tokens")
plt.axvline(512, color="#e15759", linestyle="--", label="512 tokens")
plt.title("Report Token Length Distribution")
plt.xlabel("Original RadGraph token count")
plt.ylabel("Report count")
plt.legend()
save_current_figure("report_token_lengths.png")

In [ ]:
plt.figure(figsize=(8, 4))
label_counts = entity_df["label"].value_counts().sort_values()
sns.barplot(x=label_counts.values, y=label_counts.index, color="#76b7b2")
plt.title("Entity Label Distribution")
plt.xlabel("Entity count")
plt.ylabel("Entity label")
save_current_figure("entity_label_distribution.png")

In [ ]:
plt.figure(figsize=(7, 4))
relation_counts = relation_df["label"].value_counts().sort_values()
sns.barplot(x=relation_counts.values, y=relation_counts.index, color="#edc948")
plt.title("Relation Label Distribution")
plt.xlabel("Relation count")
plt.ylabel("Relation label")
save_current_figure("relation_label_distribution.png")

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=report_df, x="entity_count", y="relation_count", hue="dataset", s=45, alpha=0.85)
plt.title("Entities and Relations per Report")
plt.xlabel("Entity count")
plt.ylabel("Relation count")
save_current_figure("report_annotation_counts.png")

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(relation_df["token_gap"], bins=30, color="#b07aa1")
plt.axvline(96, color="#e15759", linestyle="--", label="candidate distance = 96")
plt.title("Relation Argument Distance Distribution")
plt.xlabel("Token gap between relation arguments")
plt.ylabel("Relation count")
plt.legend()
save_current_figure("relation_distance_distribution.png")

In [ ]:
plt.figure(figsize=(8, 4))
category_assertion_counts = Counter(f"{row.category}::{row.assertion}" for row in entity_df.itertuples(index=False))
ordered = pd.Series(category_assertion_counts).sort_values()
sns.barplot(x=ordered.values, y=ordered.index, color="#9c755f")
plt.title("Entity Category and Assertion Distribution")
plt.xlabel("Entity count")
plt.ylabel("Category and assertion")
save_current_figure("entity_category_assertion_distribution.png")

In [ ]:
saved_figures